In [1]:
import pandas as pd
import numpy as np
import torch
from tqdm import tqdm
import joblib
from scipy import sparse
from underthesea import sent_tokenize
import networkx as nx
from sklearn.metrics.pairwise import cosine_similarity
import re
import string
from sentence_transformers import SentenceTransformer

/usr/local/lib/python3.10/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
class VietNameseSentenceTransformer():
    def __init__ (self, model_name, device='cpu', batch_size = 32):
        self.model_name = model_name
        self.batch_size = batch_size
        self.model = SentenceTransformer(model_name, device = device)

    def encode(self, sentences):
        # out put khi chạy qua model sẽ là matrix có cỡ [size(sentences), size(embdding)]
        # mỗi model sẽ có embbding size khác nhau ví dụ: 256, 512, 768
        embddings = self.model.encode(
            sentences,
            batch_size = 32,
            normalize_embeddings = True
        )
        return embddings
   

In [3]:
# đếm số lượng từ trong câu lúc tách và loại bỏ những câu có số lương từ ít
def sentence_tokenize(text, min_words=5):
    sentences = sent_tokenize(text)
    
    filtered = [
        s.strip() 
        for s in sentences 
        if len(s.split()) >= min_words
    ]
    
    return filtered

In [4]:
# xử lý câu sau khi tách
def clean_text(text):

    # loại URL
    text = re.sub(r'http\S+|www\S+', '', text)

    # chuẩn hóa khoảng trắng
    text = re.sub(r'\s+', ' ', text).strip()

    return text

In [5]:
# hàm tính page rank và lựa chọn k-top
def text_rank_summary(model, text, top_n=5, devicve='cpu'):
    # tách các câu
    sentences = sentence_tokenize(text)

    if len(sentences) <= top_n:
        return " ".join(sentences)

    # Clean từng câu
    clean_sentences = [clean_text(s) for s in sentences]

    # matrix simialrity sau model
    embeddings = model.encode(clean_sentences)

    # khi normaliz thì phép nhân matrix sẽ có kết quả tương tự cosine similarity
    similarity_matrix = np.dot(embeddings, embeddings.T)

    # bỏ self similarity
    np.fill_diagonal(similarity_matrix, 0)

     # prune cạnh yếu
    similarity_matrix[similarity_matrix < 0.3] = 0

    # Graph(tạo đồ thị giữa các câu)
    graph = nx.from_numpy_array(similarity_matrix)

    # PageRank (dùng thư viện)
    scores = nx.pagerank(graph)

    # Ranking
    ranked = sorted(
        ((scores[i], i, s) for i, s in enumerate(sentences)),
        reverse=True
    )

    # Giữ thứ tự gốc
    top_sentences = sorted(ranked[:top_n], key=lambda x: x[1])

    summary = " ".join([s for (_, _, s) in top_sentences])

    return summary

In [6]:
# Load dataset test
path = "/data/dataset_news_summary/tf_idf/test/"
file = 'test_tf_idf.json'
df = pd.read_json(path + file)

In [7]:
df.head()

,Title,Summary,Contents,Category,textrank_summary
0,Vay tiền qua app: Bí ẩn văn phòng làm việc của...,Nhiều app cho vay có văn phòng làm việc rất bí...,"Doạ chặt ngón tay ""con nợ"" Phản ánh đến Báo La...",Xã hội,"Nhanh lên chuyển qua liền ngay đây, 10 phút sa..."
1,Nông dân làm gì để vay được vốn làm nông nghiệ...,"Ngày 13.10, tại Diễn đàn Nông dân quốc gia lần...","""Sản xuất nông nghiệp công nghệ cao phải khẳng...",Kinh doanh,Ví dụ như dự án có các hợp đồng tiêu thụ ổn đị...
2,Chế độ ăn ảnh hưởng trực tiếp tới tương lai hà...,Theo một nghiên cứu mang tính bước ngoặt được ...,Một nhóm gồm 30 nhà nghiên cứu kết luận trên t...,Thế giới,Một nhóm gồm 30 nhà nghiên cứu kết luận trên t...
3,"Hà Nội ghi nhận 1.866 ca COVID-19 mới, gần 700...","Hà Nội - Theo Sở Y tế Hà Nội, từ 18h ngày 29.1...",Phân bố theo nơi ghi nhận như sau: Tại cộng đồ...,Xã hội,Trong đó 699 ca cộng đồng ghi nhận tại 233 xã ...
4,"Tin tức pháp luật 24h: Lừa đảo “chạy” việc, ng...",Nguyên hiệu trưởng nhận tổng số tiền 890 triệu...,Khởi tố hiệu trưởng nhận hơn 1 tỉ đồng lừa xin...,Pháp luật,Khởi tố người đâm chết bạn nhậu vì vỗ mông vợ ...


In [7]:
model_name = 'bkai-foundation-models/vietnamese-bi-encoder'
# config
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = VietNameseSentenceTransformer(model_name, device= device)

Loading weights: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 199/199 [00:00<00:00, 641.06it/s, Materializing param=pooler.dense.weight]


In [8]:
tqdm.pandas()
df["summary_bert"] = df["Contents"].apply(
    lambda x: text_rank_summary(model, x, top_n=5)
)

In [9]:
df.head()

,Title,Summary,Contents,Category,textrank_summary,summary_bert
0,Vay tiền qua app: Bí ẩn văn phòng làm việc của...,Nhiều app cho vay có văn phòng làm việc rất bí...,"Doạ chặt ngón tay ""con nợ"" Phản ánh đến Báo La...",Xã hội,"Nhanh lên chuyển qua liền ngay đây, 10 phút sa...","Nhanh lên chuyển qua liền ngay đây, 10 phút sa..."
1,Nông dân làm gì để vay được vốn làm nông nghiệ...,"Ngày 13.10, tại Diễn đàn Nông dân quốc gia lần...","""Sản xuất nông nghiệp công nghệ cao phải khẳng...",Kinh doanh,Ví dụ như dự án có các hợp đồng tiêu thụ ổn đị...,Ví dụ như dự án có các hợp đồng tiêu thụ ổn đị...
2,Chế độ ăn ảnh hưởng trực tiếp tới tương lai hà...,Theo một nghiên cứu mang tính bước ngoặt được ...,Một nhóm gồm 30 nhà nghiên cứu kết luận trên t...,Thế giới,Một nhóm gồm 30 nhà nghiên cứu kết luận trên t...,Một nhóm gồm 30 nhà nghiên cứu kết luận trên t...
3,"Hà Nội ghi nhận 1.866 ca COVID-19 mới, gần 700...","Hà Nội - Theo Sở Y tế Hà Nội, từ 18h ngày 29.1...",Phân bố theo nơi ghi nhận như sau: Tại cộng đồ...,Xã hội,Trong đó 699 ca cộng đồng ghi nhận tại 233 xã ...,Một số quận huyện ghi nhận nhiều bệnh nhân tro...
4,"Tin tức pháp luật 24h: Lừa đảo “chạy” việc, ng...",Nguyên hiệu trưởng nhận tổng số tiền 890 triệu...,Khởi tố hiệu trưởng nhận hơn 1 tỉ đồng lừa xin...,Pháp luật,Khởi tố người đâm chết bạn nhậu vì vỗ mông vợ ...,Khởi tố người đâm chết bạn nhậu vì vỗ mông vợ ...


In [11]:
df.to_json("/data/dataset_news_summary/tf_idf/test/test_bert.json",
              orient="records", force_ascii=False,indent=2)